In [ ]:
# LITHIUM

from mp_api.client import MPRester
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


API_KEY = "4IeHY5jVcrgiKXNuAo6Jgs7yC0Z3hsli"
try:
    with MPRester(API_KEY) as mpr:
        # Query for Li-based cathode materials with high energy density
        docs = mpr.materials.insertion_electrodes.search(
        working_ion="Li", average_voltage = (0.0, 6.5), stability_charge=(0.0, 0.5), max_delta_volume=(0,1),
        # 0.1 > stability_charge (meV) starts to be unstable
        fields=[
            "battery_id", "formula_discharge", "average_voltage"
            , "energy_grav", "energy_vol", "capacity_grav", "capacity_vol", "stability_charge", "fracA_charge", "max_delta_volume", "stability_discharge", "fracA_discharge"
        ])

        Fields = "average_voltage", "energy_grav", "energy_vol", "capacity_grav", "capacity_vol", "max_delta_volume", "stability_charge", "stability_discharge", "fracA_charge", "fracA_discharge"

        def average_field(docs, field):
            vals = [getattr(d, field) for d in docs if getattr(d, field) is not None]
            return sum(vals) / len(vals)


        df = pd.DataFrame([doc.dict() for doc in docs])
        df.head(20)

        for f in Fields:
            avg = average_field(docs, f)
            print(f"{f}: {avg}")
        print(len(df))
        df.head(20)

         # Select only the numeric columns relevant to battery performance
    cols_to_analyze = ['average_voltage', 'capacity_grav', 'energy_grav',
                        'max_delta_volume',
                       'fracA_charge', 'fracA_discharge', 'stability_charge', 'stability_discharge']
    correlation_matrix = (df[cols_to_analyze].corr(method="spearman"))
    correlation_matrix.head()
    plt.figure(figsize=(12, 9))
    sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
    plt.title("Correlation Heatmap of Battery Properties")
    plt.show()
    print(f"\nData Size: {len(df)}")



except Exception as e:
    print(f"An error occurred: {e}")




In [1]:
import numpy as np

# 1. Setup Targets
target_perf = 80
target_cost = 50

# 2. Generate Random Data (Performance and Cost from 0-100)
np.random.seed(42)
performance = np.random.uniform(0, 100, 200)
cost = np.random.uniform(0, 100, 200)

# 3. Transform: Calculate absolute distance from targets
# Instead of maximizing performance, we minimize the gap to 80.
dist_perf = np.abs(performance - target_perf)
dist_cost = np.abs(cost - target_cost)

# 4. Identify Pareto Front (Minimizing distances)
def get_pareto_front(dists):
    is_efficient = np.ones(dists.shape[0], dtype=bool)
    for i, c in enumerate(dists):
        if is_efficient[i]:
            # Keep points that are not dominated by any other point
            is_efficient[is_efficient] = np.any(dists[is_efficient] < c, axis=1)
            is_efficient[i] = True
    return is_efficient

# Combine distances and find the mask
dist_matrix = np.vstack((dist_perf, dist_cost)).T
pareto_mask = get_pareto_front(dist_matrix)

# 5. Extract original values that hit the target-based front
results = np.vstack((performance[pareto_mask], cost[pareto_mask])).T
print("Solutions that best balance being near Perf:80 and Cost:50:")
print(results[:5]) # Display first 5 optimal tradeoffs


Solutions that best balance being near Perf:80 and Cost:50:
[[80.21969808 70.2484084 ]
 [72.9007168  50.15162947]
 [72.96061783 46.55980181]
 [80.36720769 40.89529444]
 [80.74401552 54.92266647]]
